# RSCT-MAST Quickstart

Compare MAST binary failure annotations with GeoCert structured certificates.

**Requirements**: `numpy` (that's it)

## 1. Load MAST Traces

61 human-annotated multi-agent traces (31 AG2, 30 HyperAgent) from [Cemri et al. 2025](https://arxiv.org/abs/2503.13657).

In [1]:
import sys
sys.path.insert(0, '..')

from load_mast import load_all_traces, MAST_CATEGORIES, MODE_TO_CATEGORY

traces = load_all_traces()
print(f"Loaded {len(traces)} traces")
print(f"  AG2: {sum(1 for t in traces if t['source'] == 'AG2')}")
print(f"  HyperAgent: {sum(1 for t in traces if t['source'] == 'HyperAgent')}")

Loaded 61 traces
  AG2: 31
  HyperAgent: 30


## 2. Extract Structural Signals

17 scalar signals computed directly from trajectory structure. No labels, no classification — just measurement.

In [2]:
from signals import extract_signals
import numpy as np

for t in traces:
    t['signals'] = extract_signals(t)

# Show signals for first trace
print(f"Trace: {traces[0]['instance_id']} ({traces[0]['source']})")
print(f"Steps: {traces[0]['n_steps']}")
print(f"Failures: {sum(traces[0]['annotations'].values())}")
print("\nSignals:")
for k, v in sorted(traces[0]['signals'].items()):
    print(f"  {k:25s} {v:.3f}")

Trace: 02da9c1f-7c36-5739-b723-33a7d4f8e7e7 (AG2)
Steps: 10
Failures: 2

Signals:
  code_density              0.300
  consecutive_dupe_rate     0.000
  dominance                 0.500
  entropy_agents            1.000
  error_density             0.000
  growth_ratio              0.466
  handoff_rate              1.000
  has_termination           0.000
  length_cv                 1.060
  max_role_bigram           0.556
  mean_step_length          360.700
  n_agents                  2.000
  n_steps                   10.000
  repetition_ratio          0.300
  tail_termination          0.000
  total_chars               3607.000
  verify_density            0.200


## 3. Category Separation

Are MAST's three failure categories separable in signal space?

In [3]:
from collections import defaultdict

# Group traces by MAST category
cat_traces = defaultdict(list)
baseline = []

for t in traces:
    active = [MODE_TO_CATEGORY.get(m, 'unknown') for m, v in t['annotations'].items() if v]
    cats = set(active)
    if not cats:
        baseline.append(t)
    for c in cats:
        cat_traces[c].append(t)

# Signal keys (exclude n_steps which has different scale)
sig_keys = sorted(k for k in traces[0]['signals'].keys() if k != 'n_steps')

def centroid(trace_list):
    mat = np.array([[t['signals'][k] for k in sig_keys] for t in trace_list])
    return mat.mean(axis=0), mat.std(axis=0) + 1e-8

# Compute centroids
groups = {}
for name, tlist in cat_traces.items():
    groups[name] = centroid(tlist)
if baseline:
    groups['baseline'] = centroid(baseline)

# Pairwise z-score distances
names = sorted(groups.keys())
print("Pairwise centroid distances (z-score space):")
print()
for i, a in enumerate(names):
    for b in names[i+1:]:
        mu_a, std_a = groups[a]
        mu_b, std_b = groups[b]
        pooled_std = np.sqrt((std_a**2 + std_b**2) / 2)
        dist = np.sqrt(np.sum(((mu_a - mu_b) / pooled_std)**2))
        print(f"  {a:30s} vs {b:30s} : {dist:.3f}")

Pairwise centroid distances (z-score space):

  FC1_Specification              vs FC2_Misalignment               : 4.223
  FC1_Specification              vs FC3_Verification               : 1.729
  FC1_Specification              vs baseline                       : 1.235
  FC2_Misalignment               vs FC3_Verification               : 2.198
  FC2_Misalignment               vs baseline                       : 3.535
  FC3_Verification               vs baseline                       : 1.268


## 4. GeoCert Failure Taxonomy

9 fine-grained evaluation-failure modes in 3 categories. Each mode has expected signal patterns and diagnostic tests.

In [4]:
from stress_geocert import get_geocert_taxonomy

taxonomy = get_geocert_taxonomy()
for cat_id, cat in taxonomy['categories'].items():
    print(f"\n{cat_id} — {cat['name']}")
    for mode in cat['modes']:
        print(f"  {mode['id']}: {mode['name']}")
        print(f"         {mode['description'][:80]}...")


GC1 — GC1_LABEL_CONSTRUCTION
  GCF-1.1: Tercile Uniformity
         Residual-tercile labels collapse toward uniform or bland class assignments, maki...
  GCF-1.2: Label-Solver Coupling
         The label construction encodes the behavior of a particular solver rather than s...
  GCF-1.3: Target-Difficulty Conflation
         Hard targets receive worse labels because the target is difficult, not because t...

GC2 — GC2_DECOMPOSITION_REDUCTION
  GCF-2.1: Scalar Projection
         A single certificate projection such as alpha misranks solver quality by hiding ...
  GCF-2.2: Gate Compression
         A multi-axis certificate is reduced to one final gate decision, erasing meaningf...
  GCF-2.3: Range Compression
         Serious solvers occupy a narrow certificate band, limiting fine-grained discrimi...

GC3 — GC3_DEPLOYMENT_TRANSLATION
  GCF-3.1: Target-Solver Conflation
         The evaluation confuses hard-for-this-target with incompatible-for-this-solver....
  GCF-3.2: Proxy Calibrati

## 5. Injection-Detection Stress Suite

For each of the 9 failure modes: inject synthetic signals matching the mode, then diagnose. Perfect separation = 100% top-1 accuracy.

In [5]:
from stress_geocert import run_geocert_stress_suite

results = run_geocert_stress_suite(intensity=0.85, seed=3500)

print(f"Stress Suite Results:")
print(f"  Modes tested: {results['total_modes']}")
print(f"  Top-1 accuracy: {results['top1_accuracy']:.0%}")
print(f"  Top-3 accuracy: {results['top3_accuracy']:.0%}")
print()

for r in results['results']:
    match = 'PASS' if r['top1_match'] else 'FAIL'
    conf = r['candidates'][0]['confidence']
    print(f"  [{match}] {r['injected_mode']:8s} {r['injected_name']:30s} -> {r['top1_mode']:8s} (conf={conf:.3f})")

Stress Suite Results:
  Modes tested: 9
  Top-1 accuracy: 100%
  Top-3 accuracy: 100%

  [PASS] GCF-1.1  Tercile Uniformity             -> GCF-1.1  (conf=1.000)
  [PASS] GCF-1.2  Label-Solver Coupling          -> GCF-1.2  (conf=1.000)
  [PASS] GCF-1.3  Target-Difficulty Conflation   -> GCF-1.3  (conf=1.000)
  [PASS] GCF-2.1  Scalar Projection              -> GCF-2.1  (conf=1.000)
  [PASS] GCF-2.2  Gate Compression               -> GCF-2.2  (conf=1.000)
  [PASS] GCF-2.3  Range Compression              -> GCF-2.3  (conf=1.000)
  [PASS] GCF-3.1  Target-Solver Conflation       -> GCF-3.1  (conf=1.000)
  [PASS] GCF-3.2  Proxy Calibration Drift        -> GCF-3.2  (conf=1.000)
  [PASS] GCF-3.3  Fine-Routing Failure           -> GCF-3.3  (conf=1.000)


## 6. Diagnose a Custom Signal Record

Feed arbitrary signal values and get a typed failure diagnosis with confidence scores.

In [6]:
from stress_geocert import diagnose_geocert_failure

# Example: signals suggesting scalar projection failure
custom_signals = {
    'alpha_rank_gap': 0.82,
    'sigma_outlier': 0.75,
    'kappa_compression': 0.40,
    'gate_entropy': 0.15,
    'aggregate_separation': 0.60,
}

diagnosis = diagnose_geocert_failure(custom_signals, top_k=5)

print(f"Top diagnosis: {diagnosis['top1_mode']}")
print()
for c in diagnosis['candidates']:
    print(f"  {c['mode']:8s} {c['name']:30s} conf={c['confidence']:.3f}  ({c['category']})")

Top diagnosis: GCF-1.2

  GCF-1.2  Label-Solver Coupling          conf=1.000  (GC1)
  GCF-2.1  Scalar Projection              conf=1.000  (GC2)
  GCF-2.2  Gate Compression               conf=1.000  (GC2)
  GCF-3.3  Fine-Routing Failure           conf=1.000  (GC3)
  GCF-3.2  Proxy Calibration Drift        conf=0.833  (GC3)


## 7. The Dimensionality Argument

MAST uses 22 binary annotations per trace. GeoCert produces continuous n-attribute certificates per (solver x target) pair. The certificate space is fundamentally higher-dimensional than binary labeling — structured certificates can discover failure modes that annotation schemes cannot represent.

In [7]:
# Count unique annotation patterns vs continuous signal diversity
patterns = set()
for t in traces:
    pattern = tuple(sorted(m for m, v in t['annotations'].items() if v))
    patterns.add(pattern)

sig_mat = np.array([[t['signals'][k] for k in sig_keys] for t in traces])
sig_rank = np.linalg.matrix_rank(sig_mat, tol=0.01)

print(f"MAST: {len(patterns)} unique annotation patterns across {len(traces)} traces")
print(f"Signals: {len(sig_keys)} features, effective rank = {sig_rank}")
print(f"\nBinary labels collapse a continuous space into {len(patterns)} bins.")
print(f"Certificate signals preserve {sig_rank} independent axes of variation.")

MAST: 37 unique annotation patterns across 61 traces
Signals: 16 features, effective rank = 14

Binary labels collapse a continuous space into 37 bins.
Certificate signals preserve 14 independent axes of variation.
